<a href="https://colab.research.google.com/github/andreanchalvari/SkripsiAndre/blob/main/GAN_FINAL.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#TRAINING GAN

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    balanced_X,
    balanced_y,
    test_size=0.3,
    stratify=balanced_y,
    random_state=42
)

print("Train shape:", X_train.shape)
print("Test shape :", X_test.shape)

In [ ]:
@tf.function
def train_step(real_pixels, real_labels):

    real_labels = tf.expand_dims(real_labels, axis=1)

    # Train Discriminator
    with tf.GradientTape() as disc_tape:

        fake_labels = generator(real_pixels, training=True)

        real_input = tf.concat([real_pixels, real_labels], axis=1)
        fake_input = tf.concat([real_pixels, fake_labels], axis=1)

        real_output = discriminator(real_input, training=True)
        fake_output = discriminator(fake_input, training=True)

        disc_loss = (
            bce(tf.ones_like(real_output), real_output)
            +
            bce(tf.zeros_like(fake_output), fake_output)
        )

    gradients_disc = disc_tape.gradient(
        disc_loss,
        discriminator.trainable_variables
    )

    disc_optimizer.apply_gradients(
        zip(gradients_disc, discriminator.trainable_variables)
    )

    # Train Generator
    with tf.GradientTape() as gen_tape:

        fake_labels = generator(real_pixels, training=True)

        fake_input = tf.concat([real_pixels, fake_labels], axis=1)
        fake_output = discriminator(fake_input, training=True)

        # Adversarial Loss
        adv_loss = bce(
            tf.ones_like(fake_output),
            fake_output
        )

        # Reconstruction Loss
        recon_loss = bce(
            real_labels,
            fake_labels
        )

        gen_loss = recon_loss + 0.1 * adv_loss

    gradients_gen = gen_tape.gradient(
        gen_loss,
        generator.trainable_variables
    )

    gen_optimizer.apply_gradients(
        zip(gradients_gen, generator.trainable_variables)
    )

    return gen_loss, disc_loss, recon_loss, adv_loss

In [ ]:
BATCH_SIZE = 8

train_dataset = tf.data.Dataset.from_tensor_slices((X_train, y_train))
train_dataset = train_dataset.shuffle(10000).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

In [ ]:
g_history = []
d_history = []
recon_history = []
adv_history = []

In [ ]:
import os

EPOCHS = 5

for epoch in range(EPOCHS):

    g_loss_epoch = 0
    d_loss_epoch = 0
    recon_epoch = 0
    adv_epoch = 0

    steps = 0

    for pixels_batch, labels_batch in train_dataset:

        g_loss, d_loss, recon_loss, adv_loss = train_step(
            pixels_batch,
            labels_batch
        )

        g_loss_epoch += g_loss
        d_loss_epoch += d_loss
        recon_epoch += recon_loss
        adv_epoch += adv_loss

        steps += 1

    # Rata-rata loss tiap epoch
    mean_g = float(g_loss_epoch / steps)
    mean_d = float(d_loss_epoch / steps)
    mean_recon = float(recon_epoch / steps)
    mean_adv = float(adv_epoch / steps)

    g_history.append(mean_g)
    d_history.append(mean_d)
    recon_history.append(mean_recon)
    adv_history.append(mean_adv)

    print(f"\nEpoch {epoch+1}/{EPOCHS}")
    print("Generator Loss     :", mean_g)
    print("Discriminator Loss :", mean_d)
    print("Reconstruction Loss:", mean_recon)
    print("Adversarial Loss   :", mean_adv)

# Simpan model
generator.save_weights(
    os.path.join(
        MODEL_SAVE_PATH,
        "generator_final.weights.h5"
    )
)

discriminator.save_weights(
    os.path.join(
        MODEL_SAVE_PATH,
        "discriminator_final.weights.h5"
    )
)

print("\n======================================")
print("Training selesai.")
print("Model hasil epoch terakhir berhasil disimpan.")
print("Generator :", os.path.join(MODEL_SAVE_PATH, "generator_final.weights.h5"))
print("Discriminator :", os.path.join(MODEL_SAVE_PATH, "discriminator_final.weights.h5"))
print("======================================")

# TESTING GAN

In [ ]:
import numpy as np
from sklearn.metrics import confusion_matrix

print("\n=== TESTING MODEL ===")

# Prediksi probabilitas
pred_probs = generator.predict(X_test, batch_size=4096, verbose=0)

# Threshold 0.5
pred_binary = (pred_probs > 0.5).astype(np.uint8).reshape(-1)
y_true = y_test.astype(np.uint8).reshape(-1)

print("Total test samples:", len(y_true))

In [ ]:
import numpy as np

print("Jumlah data testing :", len(X_test))

# Prediksi probabilitas
y_pred_prob = generator.predict(
    X_test,
    batch_size=32,
    verbose=1
)

# Threshold menjadi label biner
y_pred = (y_pred_prob >= 0.5).astype(np.uint8)

# Flatten seluruh pixel
y_true = y_test.flatten().astype(np.uint8)
y_pred = y_pred.flatten().astype(np.uint8)

print("Jumlah Ground Truth :", len(y_true))
print("Jumlah Prediksi     :", len(y_pred))

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix

# LOAD MODEL
generator.load_weights(
    os.path.join(
        MODEL_SAVE_PATH,
        "generator_final.weights.h5"
    )
)

print("Generator final berhasil dimuat.")

# PREDIKSI DATA TESTING
pred_probs = generator.predict(
    X_test,
    batch_size=4096,
    verbose=1
)

# Konversi probabilitas menjadi label biner
pred_binary = (pred_probs >= 0.5).astype(np.uint8).reshape(-1)

# Ground Truth
y_true = y_test.astype(np.uint8).reshape(-1)

# HITUNG CONFUSION MATRIX
cm = confusion_matrix(
    y_true,
    pred_binary
)

TN, FP, FN, TP = cm.ravel()

print("\n========== CONFUSION MATRIX ==========")
print(cm)

print(f"\nTrue Negative (TN) : {TN:,}")
print(f"False Positive (FP): {FP:,}")
print(f"False Negative (FN): {FN:,}")
print(f"True Positive (TP) : {TP:,}")

# VISUALISASI CONFUSION MATRIX
plt.figure(figsize=(7,6))

ax = sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    linewidths=1,
    square=True,
    cbar=True,
    xticklabels=["Non-Cloud", "Cloud"],
    yticklabels=["Non-Cloud", "Cloud"],
    annot_kws={
        "fontsize":13,
        "fontweight":"bold"
    }
)

# Keterangan warna (Color Bar)
cbar = ax.collections[0].colorbar
cbar.set_label(
    "Jumlah Piksel",
    fontsize=12
)

# Label Sumbu
plt.xlabel(
    "Predicted Label",
    fontsize=12,
    fontweight="bold"
)

plt.ylabel(
    "True Label",
    fontsize=12,
    fontweight="bold"
)

plt.title(
    "Confusion Matrix GAN Cloud Detection",
    fontsize=14,
    fontweight="bold"
)

plt.xticks(fontsize=11)
plt.yticks(fontsize=11, rotation=0)

plt.tight_layout()

# SIMPAN GAMBAR
plt.savefig(
    os.path.join(
        RESULTS_PATH,
        "gan_confusion_matrix.png"
    ),
    dpi=300,
    bbox_inches="tight"
)

plt.show()

print("\nConfusion Matrix berhasil disimpan pada:")
print(
    os.path.join(
        RESULTS_PATH,
        "gan_confusion_matrix.png"
    )
)

In [ ]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

accuracy = accuracy_score(y_true, y_pred)

precision = precision_score(
    y_true,
    y_pred,
    zero_division=0
)

recall = recall_score(
    y_true,
    y_pred,
    zero_division=0
)

f1 = f1_score(
    y_true,
    y_pred,
    zero_division=0
)

print("\n===== METRIK EVALUASI =====")

print(f"Accuracy  : {accuracy:.4f}")
print(f"Precision : {precision:.4f}")
print(f"Recall    : {recall:.4f}")
print(f"F1 Score  : {f1:.4f}")